In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
os.environ["OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["OPENAI_DEPLOYMENT_NAME"] = "gpt-4o-mini"
os.environ["OPENAI_MODEL"] = "gpt-4o-mini"

open_api_key = os.getenv("AZURE_OPENAI_API_KEY")
if os.getenv("AZURE_OPENAI_API_KEY"):
    print(f"Azure Open API Key loaded: {open_api_key[:10]}...{open_api_key[-4:]}")
    print(f"Model: {os.getenv('OPENAI_MODEL')}")
    os.environ["OPENAI_API_KEY"] = open_api_key
else:
    print("Azure API Key not found in environment variables")
    print("Available environment variables with 'Azure':", [k for k in os.environ.keys() if 'Azure' in k.upper()])

groq_api_key = os.getenv("GROQ_API_KEY")
if os.getenv("GROQ_API_KEY"):
    print(f"GROQ API Key loaded: {groq_api_key[:10]}...{groq_api_key[-4:]}")
    os.environ["GROQ_API_KEY"] = groq_api_key
else:
    print("GROQ API Key not found in environment variables")
    print("Available environment variables with 'GROQ':", [k for k in os.environ.keys() if 'GROQ' in k.upper()])


Azure Open API Key loaded: 1Sttw3VbMy...EzVq
Model: gpt-4o-mini
GROQ API Key loaded: gsk_kJ3nEw...OOCG


In [4]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(
    model="gemma2-9b-it",
    api_key=os.getenv("GROQ_API_KEY")
)
llm_groq


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7cdebf7dda30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7cdee40ce030>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
# Import LangGraph dependencies
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated, Sequence
import operator

In [6]:
# Define state schema for LangGraph
class State(TypedDict):
    messages: Annotated[Sequence[HumanMessage | AIMessage], operator.add]

In [7]:
# Convert add function to LangChain tool
@tool
def add_tool(a: int, b: int) -> int:
    """Adds two numbers together.
    
    Args:
        a (int): First number
        b (int): Second number
    
    Returns:
        int: Sum of a and b
    """
    print("add_tool() called via LangChain tool ...")
    return a + b

# Test the tool
print("Testing the LangChain tool:")
print(add_tool.name)
print(add_tool.description)
print(add_tool.args)

Testing the LangChain tool:
add_tool
Adds two numbers together.

    Args:
        a (int): First number
        b (int): Second number

    Returns:
        int: Sum of a and b
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [8]:
# Create LangGraph chain WITHOUT tools
def chatbot_without_tools(state: State):
    """Simple chatbot node without tools"""
    response = llm_groq.invoke(state["messages"])
    return {"messages": [response]}

# Build the graph without tools
graph_without_tools = StateGraph(State)
graph_without_tools.add_node("chatbot", chatbot_without_tools)
graph_without_tools.set_entry_point("chatbot")
graph_without_tools.add_edge("chatbot", END)

# Compile the graph
app_without_tools = graph_without_tools.compile()

print("✅ LangGraph chain without tools created successfully!")

✅ LangGraph chain without tools created successfully!


In [9]:
# Test the chain WITHOUT tools
print("🧪 Testing LangGraph chain WITHOUT tools:")
print("=" * 50)

# Test with a math question
result_without_tools = app_without_tools.invoke({
    "messages": [HumanMessage(content="What is 15 + 27?")]
})

print("Question: What is 15 + 27?")
print(f"Response: {result_without_tools['messages'][-1].content}")
print()

🧪 Testing LangGraph chain WITHOUT tools:
Question: What is 15 + 27?
Response: 15 + 27 = 42 


Question: What is 15 + 27?
Response: 15 + 27 = 42 




In [10]:
# Create LangGraph chain WITH tools
tools = [add_tool]
llm_with_tools = llm_groq.bind_tools(tools)

def chatbot_with_tools(state: State):
    """Chatbot node that can use tools"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: State):
    """Decide whether to continue to tools or end"""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If there are tool calls, continue to tools
    if last_message.tool_calls:
        return "tools"
    # Otherwise, end
    return END

# Build the graph with tools
graph_with_tools = StateGraph(State)
graph_with_tools.add_node("chatbot", chatbot_with_tools)
graph_with_tools.add_node("tools", ToolNode(tools))

graph_with_tools.set_entry_point("chatbot")
graph_with_tools.add_conditional_edges("chatbot", should_continue)
graph_with_tools.add_edge("tools", "chatbot")

# Compile the graph
app_with_tools = graph_with_tools.compile()

print("✅ LangGraph chain with tools created successfully!")

✅ LangGraph chain with tools created successfully!


In [11]:
# Test the chain WITH tools
print("🧪 Testing LangGraph chain WITH tools:")
print("=" * 50)

# Test with the same math question
result_with_tools = app_with_tools.invoke({
    "messages": [HumanMessage(content="What is 15 + 27? Please use the add tool to calculate this.")]
})

print("Question: What is 15 + 27? Please use the add tool to calculate this.")
print("Messages in conversation:")
for i, msg in enumerate(result_with_tools['messages'], 1):
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  {i}. AI: [Tool call] {msg.tool_calls}")
    elif hasattr(msg, 'name'):
        print(f"  {i}. Tool ({msg.name}): {msg.content}")
    else:
        print(f"  {i}. {msg.__class__.__name__}: {msg.content}")
print()

🧪 Testing LangGraph chain WITH tools:
add_tool() called via LangChain tool ...
add_tool() called via LangChain tool ...
Question: What is 15 + 27? Please use the add tool to calculate this.
Messages in conversation:
  1. Tool (None): What is 15 + 27? Please use the add tool to calculate this.
  2. AI: [Tool call] [{'name': 'add_tool', 'args': {'a': 15, 'b': 27}, 'id': 'p6txx6gz8', 'type': 'tool_call'}]
  3. Tool (add_tool): 42
  4. Tool (None): 42

Question: What is 15 + 27? Please use the add tool to calculate this.
Messages in conversation:
  1. Tool (None): What is 15 + 27? Please use the add tool to calculate this.
  2. AI: [Tool call] [{'name': 'add_tool', 'args': {'a': 15, 'b': 27}, 'id': 'p6txx6gz8', 'type': 'tool_call'}]
  3. Tool (add_tool): 42
  4. Tool (None): 42



In [13]:
complex_result = app_with_tools.invoke({
    "messages": [HumanMessage(content="what is machine learning")]
})

print("Complex Result:")
print(complex_result['messages'][-1].content)

add_tool() called via LangChain tool ...
Complex Result:
Machine learning is the science of teaching computers to learn from data without being explicitly programmed.  It involves using algorithms to identify patterns and relationships in data, enabling computers to make predictions or decisions. 

Complex Result:
Machine learning is the science of teaching computers to learn from data without being explicitly programmed.  It involves using algorithms to identify patterns and relationships in data, enabling computers to make predictions or decisions. 

